# 1. Uso do ChatGPT para identificação de códigos clones nos resposítórios

este código foi executado para cada um dos repositórios analisados no estudo.

In [1]:
import os
from google.colab import drive
import openai

drive.mount('/content/drive')

directory_path = '/content/drive/MyDrive/repositorios/astroid-main' # Foi baixado o arquivo .zip de cada repositório e disponilizado no Google Drive para facilitar a manipulação dos aquivos.

def process_file_with_chatgpt(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()

        prompt = f'''You are a software engineering assistant specialized in pytest test clone detection.

You will receive one or more Python test files. Your task is to analyze the entire test suite and identify ONLY parametrizable Type-1 and Type-2 clones among test functions.

## Scope

Analyze only pytest test functions:
- top-level functions whose names start with `test_`;
- methods inside test classes whose names start with `test_`.

Ignore:
- docstrings;
- comments;
- type annotations;
- formatting;
- blank lines;
- line wrapping;
- parenthesis formatting.

A function-level docstring MUST NOT be considered an additional statement.

## Clone Definitions

### Type-1 Clone
Two or more test functions are Type-1 clones when their bodies are identical except for:
- comments;
- whitespace;
- formatting;
- docstrings.

### Type-2 Clone
Two or more test functions are Type-2 clones when they have the same syntactic structure and differ only in:
- literal values;
- constants;
- identifiers used as values;
- expected values;
- string literals;
- multiline string literals;
- code snippets passed as strings;
- values inside dedent("""...""");
- values inside f"""...""";
- values inside existing pytest.mark.parametrize decorators;
- line/column numeric values;
- simple argument values.

A Type-2 clone must preserve the same structural template.

## Type-3 Clone

Two or more test functions are Type-3 clones when they are structurally similar but contain meaningful structural modifications such as:
- added or removed executable statements;
- extra or missing asserts;
- different executable paths;
- different control flow;
- different loops;
- different try/except blocks;
- different with statements;
- different call sequences;
- partially modified executable blocks.

Type-3 clones are NOT safely parametrizable Type-2 clones.

Reject all Type-3 clones.

## Type-4 Clone

Two or more test functions are Type-4 clones when they are semantically similar but syntactically different.

They may:
- test the same feature;
- validate the same API;
- share the same intent;
- produce the same expected outcome;

but they use different executable structures or implementations.

Reject all Type-4 clones.

## Strict Exclusion Rules

Do NOT classify functions as Type-2 clones if any of the following occur:

1. Different number of statements in the test body.
   Example: one function has an extra assignment, assert, if, loop, with, try, or function call.

2. Different number of asserts.
   If one function has an extra assert, classify it as Type-3 or non-parametrizable, not Type-2.

3. Different control flow.
   Reject cases where one function has `if`, `else`, `for`, `while`, `with`, `try`, `raise`, or `return` structures that the others do not have.

4. Different function call structure.
   Reject if calls have different numbers of positional arguments or different keyword argument names.
   Example: `infer()` vs `infer(8, 6)` is not Type-2.

5. Different attributes or methods.
   Reject if the difference would require `getattr`, dynamic method selection, or conditional logic.
   Example: `obj.elts` vs `obj.items`, `assertTrue` vs `assertFalse`.

6. Different scopes.
   Do not group clones across different scopes:
   - global function vs class method;
   - methods in different classes;
   - functions in different modules/files, unless explicitly marked as cross-file candidates.

7. Different fixtures or function parameters.
   Reject if test functions receive different fixtures or have different signatures.

8. Incompatible decorators.
   Functions may be grouped only if their decorators are structurally equivalent.
   Different decorators, missing decorators, or different parametrization shapes should not be grouped.

9. Existing parametrized tests.
   Do not merge already-parametrized tests unless their decorators, function signatures, and bodies are structurally equivalent.

10. Semantic similarity alone is not enough.
   If functions test the same concept but use different code structure, classify them as Type-3 or semantic clones, not Type-2.

## Important Distinction

You must separate:

- "Detected clone": functions that look similar.
- "Parametrizable Type-2 clone": functions that can safely be converted into one pytest.mark.parametrize test without adding conditionals, loops, getattr, dynamic calls, or changing behavior.

Only include parametrizable Type-1 and Type-2 clones in the final clone classes.

## Classification Rules

For each candidate group, decide one of:

- Type-1 parametrizable clone
- Type-2 parametrizable clone
- Rejected: Type-3 / structural difference
- Rejected: semantic clone only
- Rejected: different scope
- Rejected: different fixtures/signatures
- Rejected: different decorators
- Rejected: different asserts/statements
- Rejected: not safely parametrizable

Only accepted Type-1 and Type-2 parametrizable clones should appear in the final clone classes.

## Output Format

Return a three-part report.

---

### Part 1: Summary Report

Provide:

- Total number of test functions analyzed
- Total number of parametrizable Type-1 clone classes
- Total number of parametrizable Type-2 clone classes
- Total number of cloned test functions included in accepted clone classes
- Total number of rejected candidate groups
- Main reasons for rejection

---

### Part 2: Accepted Parametrizable Clone Classes

For each accepted clone class:

Clone Class N:
- Clone type: Type-1 or Type-2
- Parametrizable: Yes
- Reason: explain briefly why the structure is equivalent
- Members:
  - File: `<file>`, Function: `<function>`
  - File: `<file>`, Function: `<function>`

Also list the parameters that would be extracted.

Example:

Extracted parameters:
- `parametrized_constant_0`: input value
- `parametrized_constant_1`: expected result

---

### Part 3: Rejected Candidate Groups

For each rejected group:

Rejected Group N:
- Functions:
  - File: `<file>`, Function: `<function>`
  - File: `<file>`, Function: `<function>`
- Apparent similarity:
  - explain why they looked similar
- Rejection reason:
  - choose one or more from the rejection rules
- Correct classification:
  - Type-3 / semantic clone / non-parametrizable Type-2 / not a clone

---

## Decision Guideline

Be conservative.

If you are unsure whether a group is safely parametrizable as Type-2, reject it and explain why.

Do not include Type-3, Type-4 or semantic clones in the accepted clone classes.

The goal is precision, not recall.

{content}'''

        response = openai.ChatCompletion.create(
            model="gpt-5",
            messages=[{"role": "user", "content": prompt}],
        )

        return response.choices[0].message['content'].strip()

    except Exception as e:
        return f"Error processing {file_path}: {str(e)}"


In [1]:
if os.path.exists(directory_path):
    for dirpath, dirnames, filenames in os.walk(directory_path):
        for filename in filenames:
            if filename.startswith("test_"):  # filtra os arquivos com prefixo "test_"
                file_path = os.path.join(dirpath, filename)
                relative_path = os.path.relpath(file_path, directory_path)
                print(f"Processing: {relative_path}")
                result = process_file_with_chatgpt(file_path)
                print(f"ChatGPT Response:\n{result}\n{'-'*50}")
else:
    print(f"Directory {directory_path} does not exist.")

# 2. Transformar saída da LLM em arquivos CSV

a partir de cada saída da LLM foi passado como parâmetro para gerar os arquivos CSV.

In [2]:
import re
import csv
import pandas as pd

INPUT_FILE = "saida_llm.txt"
OUTPUT_FILE = "saida_llm.csv"

rows = []

current_file = None
clone_counter = 0

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()

inside_accepted_section = False
inside_clone_class = False

current_clone_type = None
current_members = []

def save_current_class():
    global rows

    if not current_members:
        return

    class_size = len(current_members)

    for fn in current_members:
        rows.append({
            "repo": "",
            "file": current_file,
            "clone_class_id": f"CC{clone_counter:04d}",
            "clone_type": current_clone_type,
            "test_name": fn,
            "class_size": class_size,
            "llm_detectou": "YES"
        })

i = 0
while i < len(lines):

    line = lines[i].strip()

    m = re.match(r"Processing:\s*(.+)", line)
    if m:
        current_file = m.group(1)

    if "Accepted Parametrizable Clone Classes" in line:
        inside_accepted_section = True

    if "Rejected Candidate Groups" in line:
        if inside_clone_class:
            save_current_class()

        inside_accepted_section = False
        inside_clone_class = False
        current_members = []
        current_clone_type = None

    if inside_accepted_section:

        if re.search(r"Clone Class\s+\d+", line):

            if inside_clone_class:
                save_current_class()

            clone_counter += 1
            inside_clone_class = True
            current_members = []
            current_clone_type = None

        m = re.search(r"Clone type:\s*(Type-\d+)", line)
        if m:
            current_clone_type = m.group(1)

        m = re.search(r"Function:\s*`?([A-Za-z0-9_\.]+)`?", line)
        if m and inside_clone_class:
            current_members.append(m.group(1))

    i += 1

if inside_clone_class:
    save_current_class()

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "repo",
            "file",
            "clone_class_id",
            "clone_type",
            "test_name",
            "class_size",
            "llm_detectou"
        ]
    )

    writer.writeheader()
    writer.writerows(rows)

print(f"Classes encontradas: {clone_counter}")
print(f"Funções clonadas: {len(rows)}")
print(f"CSV salvo em {OUTPUT_FILE}")

# 3. Unificar as saídas em um arquivo e gerar amostra extratificada:

In [3]:
df1 = pd.read_csv("/content/astroid_llm.csv")
df1["file"] = df1["file"].str.rsplit("/", n=1).str[-1]
df1["repo"] = 'astroid'
df2 = pd.read_csv("/content/jedi_llm.csv")
df2["file"] = df2["file"].str.rsplit("/", n=1).str[-1]
df2["repo"] = 'jedi'
df3 = pd.read_csv("/content/parso_llm.csv")
df3["file"] = df3["file"].str.rsplit("/", n=1).str[-1]
df3["repo"] = 'parso'
df4 = pd.read_csv("/content/requests_llm.csv")
df4["file"] = df4["file"].str.rsplit("/", n=1).str[-1]
df4["repo"] = 'requests'

df_llm = pd.concat([df1, df2, df3, df4], ignore_index=True)
df_llm.to_csv("execucao_llm.csv")
df_llm

#calcular o tamanho da amostra
from math import ceil

N = len(classes)
Z = 1.96      # 95% confiança
p = 0.5
e = 0.05      # 5% erro

n = (N * Z**2 * p * (1-p)) / (
    e**2 * (N - 1) + Z**2 * p * (1-p)
)

#amostra por repositório
tamanho_amostra = ceil(n)

dist['amostra'] = (
    dist['qtd_classes']
    / dist['qtd_classes'].sum()
    * tamanho_amostra
).round().astype(int)

print(dist)

#amostra das classes
amostra_classes = (
    classes
    .groupby('repo', group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(
                len(x),
                dist.loc[
                    dist['repo'] == x.name,
                    'amostra'
                ].iloc[0]
            ),
            random_state=42
        )
    )
)

print(amostra_classes.head())

#recuperar a informação sobre cada classe da amostra
df_amostra = df_llm.merge(
    amostra_classes,
    on=['repo', 'file', 'clone_class_id'],
    how='inner'
)

# 4. Gerar arquivo a partir da saída do pyteor


In [4]:
import re
import pandas as pd
from pathlib import Path

def parse_pyteror_output_only_refactored(text, repo):
    rows = []

    lines = text.splitlines()
    i = 0

    while i < len(lines):
        line = lines[i].strip()

        if line.startswith("Created clone class"):
            current_functions = []
            i += 1

            while i < len(lines) and "Function" in lines[i]:
                func_line = lines[i].strip()
                func_name = func_line.replace("Function", "").strip()
                func_name = func_name.split(".")[-1]  # remove prefixo de classe
                current_functions.append(func_name)
                i += 1

            is_refactored = False
            path = None

            j = i
            while j < len(lines):
                next_line = lines[j].strip()


                if next_line.startswith("Created clone class"):
                    break

                if next_line.startswith("Refactored clone class"):
                    is_refactored = True

                if "into" in next_line and "file:" in next_line:
                    match = re.search(r'file:\s(.+)', next_line)
                    if match:
                        path = match.group(1).strip()

                j += 1

            if is_refactored and path:
                file_name = Path(path).name

                for func in current_functions:
                    rows.append({
                        "file": file_name,
                        "path": path,
                        "test_name": func,
                        "repo": repo
                    })

            i = j
        else:
            i += 1

    return pd.DataFrame(rows).drop_duplicates()



df_repositorio = parse_pyteror_output_only_refactored(text_repositorio, repositorio)
df_repositorio

# 5. Análise estatítica

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import t
from statsmodels.stats.proportion import proportion_confint


experimentos = {
    "LLM Execução 1": {
        "completo": "execucao_llm_1.csv",
        "amostra": "analise_manual_1.csv"
    },
    "LLM Execução 2": {
        "completo": "execucao_llm_2.csv",
        "amostra": "analise_manual_2.csv"
    },
    "LLM Execução 3": {
        "completo": "execucao_llm_3.csv",
        "amostra": "analise_manual_3.csv"
    }
}


def agregar_classes_completo(df):
    return (
        df.groupby(["repo", "clone_class_id"], as_index=False)
          .agg(
              clone_type=("clone_type", "first"),
              class_size=("class_size", "first"),
              qtd_funcoes=("test_name", "count")
          )
    )


def agregar_classes_amostra(df):
    df["analise_manual"] = pd.to_numeric(df["analise_manual"], errors="coerce")

    return (
        df.groupby(["repo", "clone_class_id"], as_index=False)
          .agg(
              clone_type=("clone_type", "first"),
              class_size=("class_size", "first"),
              qtd_funcoes_amostra=("test_name", "count"),
              funcoes_corretas=("analise_manual", "sum"),
              funcoes_avaliadas=("analise_manual", "count"),
              classe_correta=("analise_manual", "min")
          )
    )


def ic_wilson(acertos, total):
    return proportion_confint(
        count=acertos,
        nobs=total,
        alpha=0.05,
        method="wilson"
    )



resultados_execucoes = []
resultados_repo = []
resultados_tipo = []

classes_completas_por_execucao = {}
classes_amostra_por_execucao = {}

for nome_execucao, arquivos in experimentos.items():

    df_completo = pd.read_csv(arquivos["completo"])
    df_amostra = pd.read_csv(arquivos["amostra"])

    completo_classes = agregar_classes_completo(df_completo)
    amostra_classes = agregar_classes_amostra(df_amostra)

    classes_completas_por_execucao[nome_execucao] = completo_classes
    classes_amostra_por_execucao[nome_execucao] = amostra_classes

    total_classes_retornadas = len(completo_classes)
    classes_avaliadas = len(amostra_classes)
    classes_corretas = amostra_classes["classe_correta"].sum()
    classes_incorretas = classes_avaliadas - classes_corretas

    precisao = classes_corretas / classes_avaliadas

    ic_inf, ic_sup = ic_wilson(classes_corretas, classes_avaliadas)

    classes_corretas_estimadas = total_classes_retornadas * precisao
    estimativa_inf = total_classes_retornadas * ic_inf
    estimativa_sup = total_classes_retornadas * ic_sup

    resultados_execucoes.append({
        "execucao": nome_execucao,
        "total_classes_retornadas": total_classes_retornadas,
        "classes_avaliadas": classes_avaliadas,
        "classes_corretas_amostra": int(classes_corretas),
        "classes_incorretas_amostra": int(classes_incorretas),
        "precisao_estimada": precisao,
        "ic95_inf": ic_inf,
        "ic95_sup": ic_sup,
        "classes_corretas_estimadas": classes_corretas_estimadas,
        "classes_corretas_estimadas_inf": estimativa_inf,
        "classes_corretas_estimadas_sup": estimativa_sup
    })


    total_repo = (
        completo_classes.groupby("repo", as_index=False)
        .agg(total_classes_retornadas=("clone_class_id", "count"))
    )

    amostra_repo = (
        amostra_classes.groupby("repo", as_index=False)
        .agg(
            classes_avaliadas=("clone_class_id", "count"),
            classes_corretas=("classe_correta", "sum")
        )
    )

    repo_stats = total_repo.merge(amostra_repo, on="repo", how="left")

    repo_stats["classes_avaliadas"] = repo_stats["classes_avaliadas"].fillna(0)
    repo_stats["classes_corretas"] = repo_stats["classes_corretas"].fillna(0)

    repo_stats["precisao_estimada"] = (
        repo_stats["classes_corretas"] / repo_stats["classes_avaliadas"]
    )

    repo_stats["execucao"] = nome_execucao

    repo_stats["classes_corretas_estimadas"] = (
        repo_stats["total_classes_retornadas"] * repo_stats["precisao_estimada"]
    )

    ic_repo = repo_stats.apply(
        lambda row: ic_wilson(row["classes_corretas"], row["classes_avaliadas"])
        if row["classes_avaliadas"] > 0 else (np.nan, np.nan),
        axis=1
    )

    repo_stats["ic95_inf"] = [x[0] for x in ic_repo]
    repo_stats["ic95_sup"] = [x[1] for x in ic_repo]

    resultados_repo.append(repo_stats)


    total_tipo = (
        completo_classes.groupby("clone_type", as_index=False)
        .agg(total_classes_retornadas=("clone_class_id", "count"))
    )

    amostra_tipo = (
        amostra_classes.groupby("clone_type", as_index=False)
        .agg(
            classes_avaliadas=("clone_class_id", "count"),
            classes_corretas=("classe_correta", "sum")
        )
    )

    tipo_stats = total_tipo.merge(amostra_tipo, on="clone_type", how="left")

    tipo_stats["classes_avaliadas"] = tipo_stats["classes_avaliadas"].fillna(0)
    tipo_stats["classes_corretas"] = tipo_stats["classes_corretas"].fillna(0)

    tipo_stats["precisao_estimada"] = (
        tipo_stats["classes_corretas"] / tipo_stats["classes_avaliadas"]
    )

    tipo_stats["execucao"] = nome_execucao

    tipo_stats["classes_corretas_estimadas"] = (
        tipo_stats["total_classes_retornadas"] * tipo_stats["precisao_estimada"]
    )

    ic_tipo = tipo_stats.apply(
        lambda row: ic_wilson(row["classes_corretas"], row["classes_avaliadas"])
        if row["classes_avaliadas"] > 0 else (np.nan, np.nan),
        axis=1
    )

    tipo_stats["ic95_inf"] = [x[0] for x in ic_tipo]
    tipo_stats["ic95_sup"] = [x[1] for x in ic_tipo]

    resultados_tipo.append(tipo_stats)


df_execucoes = pd.DataFrame(resultados_execucoes)
df_repo = pd.concat(resultados_repo, ignore_index=True)
df_tipo = pd.concat(resultados_tipo, ignore_index=True)

print("\nRESULTADO POR EXECUÇÃO")
print(df_execucoes)

print("\nRESULTADO POR REPOSITÓRIO")
print(df_repo)

print("\nRESULTADO POR TIPO DE CLONE")
print(df_tipo)



precisoes = df_execucoes["precisao_estimada"]

media = precisoes.mean()
dp = precisoes.std(ddof=1)
coef_var = dp / media

erro = t.ppf(0.975, len(precisoes) - 1) * dp / np.sqrt(len(precisoes))

df_resumo_llm = pd.DataFrame([{
    "metodo": "LLM média das 3 execuções",
    "precisao_media": media,
    "desvio_padrao": dp,
    "coeficiente_variacao": coef_var,
    "ic95_media_inf": media - erro,
    "ic95_media_sup": media + erro,
    "media_classes_retornadas": df_execucoes["total_classes_retornadas"].mean(),
    "media_classes_corretas_estimadas": df_execucoes["classes_corretas_estimadas"].mean()
}])

print("\nRESUMO DAS TRÊS EXECUÇÕES")
print(df_resumo_llm)


df_execucoes.to_csv("resultado_llm_por_execucao.csv", index=False)
df_repo.to_csv("resultado_llm_por_repositorio.csv", index=False)
df_tipo.to_csv("resultado_llm_por_tipo_clone.csv", index=False)
df_resumo_llm.to_csv("resumo_llm_3_execucoes.csv", index=False)



# 6. Classes identificadas pela LLM e pelo PyTeRor por repositório

In [ ]:
import pandas as pd


llm_exec1 = "execucao_llm_1.csv"
llm_exec2 = "execucao_llm_2.csv"
llm_exec3 = "execucao_llm_3.csv"
pyteror   = "pyteror_saida.csv"


def contar_classes(csv):

    df = pd.read_csv(csv)

    resumo = (
        df.groupby("repo")["clone_class_id"]
          .nunique()
          .reset_index(name="classes")
    )

    return resumo


e1 = contar_classes(llm_exec1).rename(columns={"classes":"LLM E1"})
e2 = contar_classes(llm_exec2).rename(columns={"classes":"LLM E2"})
e3 = contar_classes(llm_exec3).rename(columns={"classes":"LLM E3"})
pt = contar_classes(pyteror).rename(columns={"classes":"PyTeRor"})


tabela = (
    e1.merge(e2, on="repo", how="outer")
      .merge(e3, on="repo", how="outer")
      .merge(pt, on="repo", how="outer")
      .fillna(0)
)


for c in ["LLM E1","LLM E2","LLM E3","PyTeRor"]:
    tabela[c] = tabela[c].astype(int)


tabela["Média LLM"] = (
    tabela[["LLM E1","LLM E2","LLM E3"]]
    .mean(axis=1)
)


def diferenca(row):

    if row["PyTeRor"] == 0:
        return None

    return ((row["Média LLM"]-row["PyTeRor"])/row["PyTeRor"])*100

tabela["Diferença (%)"] = tabela.apply(diferenca, axis=1)


total = pd.DataFrame({

    "repo":["TOTAL"],

    "LLM E1":[tabela["LLM E1"].sum()],
    "LLM E2":[tabela["LLM E2"].sum()],
    "LLM E3":[tabela["LLM E3"].sum()],

    "Média LLM":[tabela["Média LLM"].sum()],

    "PyTeRor":[tabela["PyTeRor"].sum()],

    "Diferença (%)":[
        (
            (tabela["Média LLM"].sum()-
             tabela["PyTeRor"].sum())
            /
            tabela["PyTeRor"].sum()
        )*100
    ]

})

tabela = pd.concat([tabela,total],ignore_index=True)


tabela["Média LLM"] = tabela["Média LLM"].round(1)
tabela["Diferença (%)"] = tabela["Diferença (%)"].round(1)

print(tabela)

tabela.to_csv("Tabela4_Cobertura.csv",index=False)

# 7. Calculo do ARI e NMI


In [ ]:
def criar_rotulos_para_comparacao(df_a, df_b):
    df_a = preparar_df(df_a)
    df_b = preparar_df(df_b)

    funcoes_comuns = sorted(
        set(df_a["funcao_id"]) & set(df_b["funcao_id"])
    )

    mapa_a = (
        df_a.drop_duplicates("funcao_id")
            .set_index("funcao_id")["classe_global"]
            .to_dict()
    )

    mapa_b = (
        df_b.drop_duplicates("funcao_id")
            .set_index("funcao_id")["classe_global"]
            .to_dict()
    )

    labels_a = [mapa_a[f] for f in funcoes_comuns]
    labels_b = [mapa_b[f] for f in funcoes_comuns]

    return funcoes_comuns, labels_a, labels_b


resultados_cluster = []

for nome_execucao, arquivo in arquivos_llm.items():
    df_llm = pd.read_csv(arquivo)

    funcoes_comuns, labels_llm, labels_py = criar_rotulos_para_comparacao(
        df_llm,
        df_pyteror
    )

    ari = adjusted_rand_score(labels_llm, labels_py)
    nmi = normalized_mutual_info_score(labels_llm, labels_py)

    resultados_cluster.append({
        "execucao": nome_execucao,
        "funcoes_comuns": len(funcoes_comuns),
        "adjusted_rand_index": ari,
        "normalized_mutual_information": nmi
    })

df_resultados_cluster = pd.DataFrame(resultados_cluster)

print(df_resultados_cluster)

df_resultados_cluster.to_csv(
    "comparacao_cluster_llm_pyteror_ari_nmi.csv",
    index=False
)